# Qwen3.5-0.8B + FlyFFN-v2 — progressive FFN-only experiment

This notebook applies the successful FlyFFN-v2 idea to **Qwen/Qwen3.5-0.8B**. Qwen3.5 token mixers (Gated DeltaNet + full attention) stay untouched; only FFNs are progressively converted. Every fourth FFN remains a dense anchor.

After training, the notebook runs a 50-item FastEval and opens a **dual interactive chat** so you can ask the same multi-turn questions to original Qwen3.5 and FlyFFN-v2 side by side.


In [1]:
#@title 1. Update repository, install current Transformers, and syntax-check
import pathlib, subprocess, sys
REPO_DIR = pathlib.Path('/content/TinyCeNN-LM')
if REPO_DIR.exists():
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin'], check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','origin/main'], check=True)
else:
    subprocess.run(['git','clone','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO_DIR)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-U','transformers','accelerate','datasets','huggingface_hub','safetensors','ipywidgets','pandas','requests','tqdm'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR)], check=True)
SRC_DIR = REPO_DIR/'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print('✓ Added to current kernel path:', SRC_DIR)
for p in [REPO_DIR/'src'/'tinycenn_lm'/'qwen35_flyffn_v2.py', REPO_DIR/'src'/'tinycenn_lm'/'qwen35_standalone.py', REPO_DIR/'scripts'/'run_qwen35_flyffn_v2.py']:
    subprocess.run([sys.executable,'-m','py_compile',str(p)], check=True)
print('✓ Qwen3.5 FlyFFN-v2 module and runner syntax OK')
import importlib.util
_spec=importlib.util.spec_from_file_location('qwen35_runner_preflight', REPO_DIR/'scripts'/'run_qwen35_flyffn_v2.py')
_runner=importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_runner)
assert hasattr(_runner.base.base,'set_seed') and hasattr(_runner.base.base,'choose_dtype')
assert all(hasattr(_runner.base,x) for x in ['stage_schedule','calibrate_progressively','global_train','dense_equivalence','v2_state','load_v2_state'])
from tinycenn_lm.qwen35_standalone import export_standalone, load_standalone, upload_standalone
print('✓ Runtime helper + standalone export wiring OK')
print('Ready:', REPO_DIR)


✓ Added to current kernel path: /content/TinyCeNN-LM/src
✓ Qwen3.5 FlyFFN-v2 module and runner syntax OK
✓ Runtime helper + standalone export wiring OK
Ready: /content/TinyCeNN-LM


In [2]:
#@title 2. Configuration
RUN_MODE = 'strong' #@param ['quick','strong']
SEQ_LEN = 128 #@param {type:'integer'}
BATCH_SIZE = 1 #@param {type:'integer'}
FLY_NODES = 256 #@param {type:'integer'}
ROUTER_RANK = 96 #@param {type:'integer'}
MAX_EDGES = 2048 #@param {type:'integer'}
NUM_SHARDS = 8 #@param {type:'integer'}
GRAPH_STEPS = 1 #@param {type:'integer'}
GRAPH_MIX_INIT = 0.50 #@param {type:'number'}
ANCHOR_EVERY = 4 #@param {type:'integer'}
MAX_CE_GAP = 0.30 #@param {type:'number'}
RUN_REWIRED_CONTROL = False #@param {type:'boolean'}
OUTPUT_DIR = REPO_DIR/'results'/'flyffn_v2_qwen35_08b'
print('Base: Qwen/Qwen3.5-0.8B (post-trained chat model)')
print('Token mixers: unchanged')
print('FFN schedule: 8→6→4→3→2, dense anchor every', ANCHOR_EVERY, 'layers')
print('Quality gate CE gap <=', MAX_CE_GAP)
print('Rewired control:', RUN_REWIRED_CONTROL)

# Standalone Hugging Face export
SAVE_STANDALONE = True #@param {type:'boolean'}
UPLOAD_TO_HF = True #@param {type:'boolean'}
HF_REPO_ID = "vtava/Qwen35-0.8B-FlyFFN-v2" #@param {type:'string'}
HF_PRIVATE = False #@param {type:'boolean'}

print('Standalone export:', SAVE_STANDALONE)
print('Upload to HF:', UPLOAD_TO_HF, '| repo:', HF_REPO_ID)


Base: Qwen/Qwen3.5-0.8B (post-trained chat model)
Token mixers: unchanged
FFN schedule: 8→6→4→3→2, dense anchor every 4 layers
Quality gate CE gap <= 0.3
Rewired control: False
Standalone export: True
Upload to HF: True | repo: vtava/Qwen35-0.8B-FlyFFN-v2


In [3]:
#@title 3. Train / evaluate Qwen3.5 FlyFFN-v2 — live output
import os, subprocess, sys
cmd=[sys.executable,'-u',str(REPO_DIR/'scripts'/'run_qwen35_flyffn_v2.py'),
     '--run-mode',RUN_MODE,'--seq-len',str(SEQ_LEN),'--batch-size',str(BATCH_SIZE),
     '--fly-nodes',str(FLY_NODES),'--router-rank',str(ROUTER_RANK),'--max-edges',str(MAX_EDGES),
     '--num-shards',str(NUM_SHARDS),'--graph-steps',str(GRAPH_STEPS),'--graph-mix-init',str(GRAPH_MIX_INIT),
     '--anchor-every',str(ANCHOR_EVERY),'--max-ce-gap',str(MAX_CE_GAP),'--output-dir',str(OUTPUT_DIR)]
if RUN_REWIRED_CONTROL: cmd.append('--rewired')
if SAVE_STANDALONE:
    cmd += ['--standalone-dir', str(OUTPUT_DIR/'standalone')]
if UPLOAD_TO_HF:
    if not HF_REPO_ID.strip():
        raise ValueError('HF_REPO_ID is empty')
    from huggingface_hub import get_token, notebook_login
    token = os.environ.get('HF_TOKEN')
    try:
        from google.colab import userdata
        if not token:
            token = userdata.get('HF_TOKEN')
    except Exception:
        pass
    if not token:
        token = get_token()
    if not token:
        notebook_login()
        token = get_token()
    if not token:
        raise RuntimeError('Hugging Face login failed')
    os.environ['HF_TOKEN'] = token
    cmd += ['--upload-hf', '--hf-repo-id', HF_REPO_ID]
    if HF_PRIVATE:
        cmd.append('--hf-private')
print('='*100)
print('Qwen3.5-0.8B / FlyFFN-v2 progressive FFN experiment')
print('Command:', ' '.join(cmd)); print('='*100)
env=os.environ.copy(); env['PYTHONUNBUFFERED']='1'; env['TQDM_MININTERVAL']='1'
if UPLOAD_TO_HF and os.environ.get('HF_TOKEN'):
    env['HF_TOKEN'] = os.environ['HF_TOKEN']
p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
for line in iter(p.stdout.readline,''):
    print(line,end='',flush=True)
rc=p.wait(); print('\nFinished, exit code',rc)
if rc: raise subprocess.CalledProcessError(rc,cmd)


Qwen3.5-0.8B / FlyFFN-v2 progressive FFN experiment
Command: /usr/bin/python3 -u /content/TinyCeNN-LM/scripts/run_qwen35_flyffn_v2.py --run-mode strong --seq-len 128 --batch-size 1 --fly-nodes 256 --router-rank 96 --max-edges 2048 --num-shards 8 --graph-steps 1 --graph-mix-init 0.5 --anchor-every 4 --max-ce-gap 0.3 --output-dir /content/TinyCeNN-LM/results/flyffn_v2_qwen35_08b --standalone-dir /content/TinyCeNN-LM/results/flyffn_v2_qwen35_08b/standalone --upload-hf --hf-repo-id vtava/Qwen35-0.8B-FlyFFN-v2
DEVICE cuda | dtype=torch.bfloat16 | gpu=NVIDIA L4
STAGE graph: preparing FlyWire topology

FlyWire graph: 0.00B [00:00, ?B/s]
FlyWire graph: 449kB [00:01, 444kB/s]
FlyWire graph: 1.04MB [00:02, 484kB/s]
FlyWire graph: 1.65MB [00:03, 509kB/s]
FlyWire graph: 2.24MB [00:04, 509kB/s]
FlyWire graph: 2.86MB [00:05, 519kB/s]
FlyWire graph: 3.51MB [00:06, 534kB/s]
FlyWire graph: 4.10MB [00:07, 526kB/s]
FlyWire graph: 4.72MB [00:09, 530kB/s]
FlyWire graph: 5.43MB [00:10, 559kB/s]
FlyWire grap

In [4]:
#@title 4. Training results
import json, pandas as pd
from IPython.display import display
summary=pd.read_csv(OUTPUT_DIR/'summary.csv',index_col=0)
report=json.loads((OUTPUT_DIR/'report.json').read_text())
chat_samples=json.loads((OUTPUT_DIR/'chat_samples.json').read_text())
display(summary)
print('\nCHECKS')
print('Architecture:',report['architecture'])
print('Token mixers unchanged:',report['token_mixers_unchanged'])
print('FlyFFN-v2 layers:',report['flyffn_layers'])
print('Dense anchors:',report['dense_anchor_layers'])
print('Dense-equivalence max logit diff:',report['dense_equivalence_biological_max_abs_logit_diff'])
print('Device:',report['device'],'| dtype:',report['dtype'])
print('\nKEY METRICS')
for k in ['fly_ce_gap_vs_qwen','fly_ppl_ratio_vs_qwen','parameter_ratio_fly_over_qwen','decode_speed_ratio_fly_over_qwen','biological_topology_ce_gain','biological_topology_ppl_gain_pct']:
    if k in report: print(k,':',report[k])
print('\nFINAL ROUTING')
for layer,state in report['biological_routing_schedule'].items():
    print(f'layer {int(layer):>2}: k={state["active_k"]}, mix={state["route_mix"]:.2f}')
print('\nTRAINING CHAT SAMPLES')
for x in chat_samples:
    print('='*90); print('USER:',x['prompt']); print('FLY:',x['reply'])


,ce,perplexity,weight_mb,buffer_mb,prefill_tokens_s,prefill_peak_extra_mb,decode_tokens_s,decode_peak_extra_mb,teacher_kl
Qwen3.5-0.8B,2.986667,19.819505,1435.075806,0.000244,1519.680029,81.21875,20.47644,3.919434,NaN
Qwen3.5 FlyFFN-v2 biological,3.012174,20.331554,1443.654068,0.250416,1148.113370,81.21875,13.81571,3.919434,0.169979



CHECKS
Architecture: Qwen3.5-0.8B + progressive FlyWire sparse FFN v2 + dense anchors
Token mixers unchanged: True
FlyFFN-v2 layers: 18
Dense anchors: [3, 7, 11, 15, 19, 23]
Dense-equivalence max logit diff: 0.40625
Device: cuda | dtype: torch.bfloat16

KEY METRICS
fly_ce_gap_vs_qwen : 0.02550750970840454
fly_ppl_ratio_vs_qwen : 1.02583560996782
parameter_ratio_fly_over_qwen : 1.0029887836918594
decode_speed_ratio_fly_over_qwen : 0.6747125372244265

FINAL ROUTING
layer  0: k=4, mix=0.50
layer  1: k=4, mix=0.50
layer  2: k=4, mix=0.50
layer  4: k=4, mix=0.50
layer  5: k=4, mix=0.50
layer  6: k=4, mix=0.50
layer  8: k=4, mix=0.50
layer  9: k=4, mix=0.50
layer 10: k=4, mix=0.50
layer 12: k=4, mix=0.25
layer 13: k=4, mix=0.25
layer 14: k=4, mix=0.25
layer 16: k=4, mix=0.25
layer 17: k=4, mix=0.25
layer 18: k=4, mix=0.25
layer 20: k=6, mix=0.10
layer 21: k=6, mix=0.10
layer 22: k=6, mix=0.10

TRAINING CHAT SAMPLES
USER: Explain why the sky is blue in a way a 10-year-old can understand.
FLY

In [5]:
#@title 5. Load original Qwen3.5 + trained FlyFFN-v2 for FastEval and chat
import gc, json, sys
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tinycenn_lm.qwen35_flyffn_v2 import FlyFFNV2Config, replace_ffns_with_fly_v2, assert_qwen35_flyffn_v2

BASE_MODEL='Qwen/Qwen3.5-0.8B'
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype=torch.bfloat16 if device.type=='cuda' and torch.cuda.is_bf16_supported() else (torch.float16 if device.type=='cuda' else torch.float32)
tokenizer=AutoTokenizer.from_pretrained(BASE_MODEL,use_fast=True)
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
report=json.loads((OUTPUT_DIR/'report.json').read_text())
state=torch.load(OUTPUT_DIR/'biological_qwen35_flyffn_v2.pt',map_location='cpu',weights_only=True)
adj=state['flyffn_shared_graph.adjacency'].float()
c=report['config']
cfg=FlyFFNV2Config(fly_nodes=int(c['fly_nodes']),router_rank=int(c['router_rank']),num_shards=int(c['num_shards']),graph_steps=int(c['graph_steps']),graph_mix_init=float(c['graph_mix_init']),anchor_every=int(c['anchor_every']))

def load_base():
    return AutoModelForCausalLM.from_pretrained(BASE_MODEL,dtype=dtype if device.type=='cuda' else torch.float32,low_cpu_mem_usage=True).to(device).eval()

print('Loading original Qwen3.5-0.8B ...')
qwen_model=load_base()
print('Loading FlyFFN-v2 Qwen3.5 ...')
fly_model=load_base(); replace_ffns_with_fly_v2(fly_model,cfg,adj)
inc=fly_model.load_state_dict(state,strict=False)
missing=[k for k in inc.missing_keys if '.mlp.' in k or k.startswith('flyffn_shared_graph.')]
if missing: raise RuntimeError('Missing FlyFFN-v2 keys: '+str(missing[:10]))
assert_qwen35_flyffn_v2(fly_model,cfg.anchor_every)
print('✓ Both models ready on',device,'|',dtype)


Loading original Qwen3.5-0.8B ...


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loading FlyFFN-v2 Qwen3.5 ...


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

✓ Both models ready on cuda | torch.bfloat16


In [6]:
#@title 6. FastEval — 50 items: MMLU-Pro / PIQA / MMMLU-DE (GPQA optional)
import os, random, time
import pandas as pd, torch
from datasets import load_dataset
from IPython.display import display
N=50; EVAL_BATCH=4; MAX_LENGTH=1024; SEED=2026
LETTERS=list('ABCDEFGHIJ')

def sample(ds,n,seed): return ds.shuffle(seed=seed).select(range(min(n,len(ds))))
def prompt_mc(q,opts,german=False):
    labels=LETTERS[:len(opts)]
    lead=('Wähle die richtige Antwort. Antworte nur mit dem Buchstaben.' if german else 'Choose the correct answer. Reply only with the answer letter.')
    lines=[lead,'',('Frage: ' if german else 'Question: ')+str(q),'']+[f'{a}. {o}' for a,o in zip(labels,opts)]+['',('Antwort:' if german else 'Answer:')]
    return '\n'.join(lines),labels

def chat_wrap(text):
    return tokenizer.apply_chat_template([{'role':'user','content':text}],tokenize=False,add_generation_prompt=True)

def label_ids(labels):
    out=[]
    for a in labels:
        choices=[tokenizer.encode(' '+a,add_special_tokens=False),tokenizer.encode(a,add_special_tokens=False)]
        one=next((x[0] for x in choices if len(x)==1),None)
        if one is None: raise RuntimeError(f'Answer label {a} is not one token: {choices}')
        out.append(one)
    return out

benches={}
ds=sample(load_dataset('TIGER-Lab/MMLU-Pro',split='test'),N,SEED)
benches['MMLU-Pro']=[{'prompt':prompt_mc(x['question'],list(x['options']))[0],'labels':prompt_mc(x['question'],list(x['options']))[1],'gold':int(x['answer_index'])} for x in ds]
ds=sample(load_dataset('regisss/piqa',split='validation'),N,SEED+1)
benches['PIQA']=[{'prompt':prompt_mc(x['goal'],[x['sol1'],x['sol2']])[0],'labels':['A','B'],'gold':int(x['label'])} for x in ds]
try: ds=load_dataset('openai/MMMLU','DE_DE',split='test')
except Exception: ds=load_dataset('openai/MMMLU',split='test')
ds=sample(ds,N,SEED+2); mm=[]
for x in ds:
    opts=[str(x[k]) for k in ['A','B','C','D']]; p,l=prompt_mc(str(x['Question']),opts,True); mm.append({'prompt':p,'labels':l,'gold':l.index(str(x['Answer']).strip().upper())})
benches['MMMLU-DE']=mm
HF_TOKEN=os.environ.get('HF_TOKEN')
try:
    from google.colab import userdata
    if not HF_TOKEN: HF_TOKEN=userdata.get('HF_TOKEN')
except: pass
try:
    ds=sample(load_dataset('Idavidrein/gpqa','gpqa_diamond',split='train',token=HF_TOKEN),N,SEED+3); gp=[]
    for i,x in enumerate(ds):
        raw=[x['Correct Answer'],x['Incorrect Answer 1'],x['Incorrect Answer 2'],x['Incorrect Answer 3']]; order=list(range(4)); random.Random(SEED+10000+i).shuffle(order)
        opts=[raw[j] for j in order]; p,l=prompt_mc(x['Question'],opts); gp.append({'prompt':p,'labels':l,'gold':order.index(0)})
    benches['GPQA-Diamond']=gp
except Exception as e: print('GPQA skipped:',str(e)[:140])

@torch.inference_mode()
def run_eval(model,name):
    ans={}; model.eval(); tokenizer.padding_side='right'; tokenizer.truncation_side='left'
    print('\n'+'='*92+'\nMODEL:',name+'\n'+'='*92)
    for bn,items in benches.items():
        correct=done=0; t0=time.perf_counter()
        for s in range(0,len(items),EVAL_BATCH):
            batch=items[s:s+EVAL_BATCH]; texts=[chat_wrap(x['prompt']) for x in batch]
            enc=tokenizer(texts,return_tensors='pt',padding=True,truncation=True,max_length=MAX_LENGTH).to(device)
            last=enc.attention_mask.sum(1)-1
            out=model(**enc,use_cache=False,return_dict=True).logits.float()
            for j,x in enumerate(batch):
                ids=torch.tensor(label_ids(x['labels']),device=device); pred=int(out[j,int(last[j])][ids].argmax())
                correct+=pred==x['gold']; done+=1
            if done%10==0 or done==len(items): print(f'  {done:>2}/{len(items)} | correct={correct:>2} | acc={100*correct/done:5.1f}% | {done/max(time.perf_counter()-t0,1e-9):5.2f} q/s')
        ans[bn]={'correct':correct,'total':done,'accuracy':correct/done}; print(f'  DONE → {correct}/{done} = {100*correct/done:.1f}%')
    return ans

base_eval=run_eval(qwen_model,'Qwen3.5-0.8B')
fly_eval=run_eval(fly_model,'Qwen3.5 FlyFFN-v2')
rows=[]
for bn in benches:
    b,f=base_eval[bn],fly_eval[bn]; bp,fp=100*b['accuracy'],100*f['accuracy']
    rows.append({'Benchmark':bn,'Qwen correct':f"{b['correct']}/{b['total']}",'Qwen %':bp,'FlyFFN correct':f"{f['correct']}/{f['total']}",'FlyFFN %':fp,'Δ FlyFFN':fp-bp})
fast_eval_df=pd.DataFrame(rows).set_index('Benchmark')
display(fast_eval_df.style.format({'Qwen %':'{:.1f}%','FlyFFN %':'{:.1f}%','Δ FlyFFN':'{:+.1f}'}))
print(f"Macro: Qwen={fast_eval_df['Qwen %'].mean():.1f}% | Fly={fast_eval_df['FlyFFN %'].mean():.1f}% | Δ={fast_eval_df['Δ FlyFFN'].mean():+.1f} points")
fast_eval_df.to_csv(OUTPUT_DIR/'fast_eval_50_qwen35.csv')


README.md:   0%|          | 0.00/11.7k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.14MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 42.9kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

GPQA skipped: Dataset 'Idavidrein/gpqa' is a gated dataset on the Hub. Visit the dataset page at https://huggingface.co/datasets/Idavidrein/gpqa to ask fo

MODEL: Qwen3.5-0.8B


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


  20/50 | correct= 1 | acc=  5.0% | 18.87 q/s
  40/50 | correct= 8 | acc= 20.0% | 23.08 q/s
  50/50 | correct=11 | acc= 22.0% | 22.89 q/s
  DONE → 11/50 = 22.0%
  20/50 | correct=12 | acc= 60.0% | 49.65 q/s
  40/50 | correct=25 | acc= 62.5% | 50.31 q/s
  50/50 | correct=32 | acc= 64.0% | 47.57 q/s
  DONE → 32/50 = 64.0%
  20/50 | correct= 8 | acc= 40.0% | 28.75 q/s
  40/50 | correct=18 | acc= 45.0% | 26.28 q/s
  50/50 | correct=22 | acc= 44.0% | 28.03 q/s
  DONE → 22/50 = 44.0%

MODEL: Qwen3.5 FlyFFN-v2
  20/50 | correct= 0 | acc=  0.0% | 26.69 q/s
  40/50 | correct= 3 | acc=  7.5% | 25.98 q/s
  50/50 | correct= 6 | acc= 12.0% | 24.29 q/s
  DONE → 6/50 = 12.0%
  20/50 | correct=13 | acc= 65.0% | 39.27 q/s
  40/50 | correct=25 | acc= 62.5% | 40.19 q/s
  50/50 | correct=32 | acc= 64.0% | 38.81 q/s
  DONE → 32/50 = 64.0%
  20/50 | correct= 6 | acc= 30.0% | 24.26 q/s
  40/50 | correct=13 | acc= 32.5% | 22.59 q/s
  50/50 | correct=16 | acc= 32.0% | 23.84 q/s
  DONE → 16/50 = 32.0%


,Qwen correct,Qwen %,FlyFFN correct,FlyFFN %,Δ FlyFFN
Benchmark,,,,,
MMLU-Pro,11/50,22.0%,6/50,12.0%,-10.0
PIQA,32/50,64.0%,32/50,64.0%,+0.0
MMMLU-DE,22/50,44.0%,16/50,32.0%,-12.0


Macro: Qwen=43.3% | Fly=36.0% | Δ=-7.3 points


In [7]:
#@title 7. Interactive dual CHAT — original Qwen3.5 vs FlyFFN-v2
import time, html, torch, ipywidgets as widgets
from IPython.display import display, HTML

base_history=[]; fly_history=[]

def _reply(model,history,user_text,max_new=192):
    msgs=history+[{'role':'user','content':user_text}]
    text=tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
    enc=tokenizer(text,return_tensors='pt').to(device)
    t=time.perf_counter()
    with torch.inference_mode():
        out=model.generate(**enc,max_new_tokens=max_new,do_sample=False,use_cache=True,pad_token_id=tokenizer.eos_token_id)
    dt=time.perf_counter()-t
    reply=tokenizer.decode(out[0,enc.input_ids.shape[1]:],skip_special_tokens=True).strip()
    return reply,dt

def ask_both(user_text):
    global base_history, fly_history
    b,bt=_reply(qwen_model,base_history,user_text); f,ft=_reply(fly_model,fly_history,user_text)
    base_history += [{'role':'user','content':user_text},{'role':'assistant','content':b}]
    fly_history  += [{'role':'user','content':user_text},{'role':'assistant','content':f}]
    return b,bt,f,ft

def reset_chat(*_):
    global base_history,fly_history
    base_history=[]; fly_history=[]; out.clear_output()
    with out: print('Chat reset ✓')

prompt=widgets.Textarea(value='Explain in simple terms how a sparse FFN can save computation, and give one possible risk.',description='You:',layout=widgets.Layout(width='100%',height='100px'))
ask=widgets.Button(description='Ask both models',button_style='success'); reset=widgets.Button(description='Reset chat')
out=widgets.Output()

def on_ask(_):
    q=prompt.value.strip()
    if not q: return
    with out:
        print('\n'+'='*100); print('USER:',q)
        b,bt,f,ft=ask_both(q)
        table=(f"<table style='width:100%;table-layout:fixed'><tr><th>Original Qwen3.5-0.8B ({bt:.2f}s)</th>"
               f"<th>FlyFFN-v2 ({ft:.2f}s)</th></tr><tr><td style='vertical-align:top;white-space:pre-wrap;padding:12px'>"
               f"{html.escape(b)}</td><td style='vertical-align:top;white-space:pre-wrap;padding:12px'>{html.escape(f)}</td></tr></table>")
        display(HTML(table))
    prompt.value=''

ask.on_click(on_ask); reset.on_click(reset_chat)
display(widgets.VBox([prompt,widgets.HBox([ask,reset]),out]))
print('Dual multi-turn chat ready. Each model keeps its own conversation history.')


Dual multi-turn chat ready. Each model keeps its own conversation history.


In [8]:
#@title 8. Verify / repair standalone export and upload (NO retraining needed)
import gc, json, os
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tinycenn_lm.qwen35_flyffn_v2 import FlyFFNV2Config, replace_ffns_with_fly_v2, assert_qwen35_flyffn_v2
from tinycenn_lm.qwen35_standalone import export_standalone, upload_standalone

STANDALONE_DIR = OUTPUT_DIR / 'standalone'
manifest_path = STANDALONE_DIR / 'standalone_manifest.json'
state_path = STANDALONE_DIR / 'standalone_state.pt'

# Repair older/broken exports without re-running training.
if not (manifest_path.exists() and state_path.exists()):
    print('Verified standalone package not found — rebuilding from trained checkpoint...')
    report = json.loads((OUTPUT_DIR/'report.json').read_text())
    c = report['config']
    cfg = FlyFFNV2Config(
        fly_nodes=int(c['fly_nodes']),
        router_rank=int(c['router_rank']),
        num_shards=int(c['num_shards']),
        graph_steps=int(c['graph_steps']),
        graph_mix_init=float(c['graph_mix_init']),
        anchor_every=int(c['anchor_every']),
    )

    if 'fly_model' in globals():
        export_model = fly_model
        export_tokenizer = tokenizer
        print('Using FlyFFN-v2 model already in notebook memory.')
    else:
        ckpt = torch.load(
            OUTPUT_DIR/'biological_qwen35_flyffn_v2.pt',
            map_location='cpu',
            weights_only=True,
        )
        adjacency = ckpt['flyffn_shared_graph.adjacency'].float()
        _device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        _dtype = torch.bfloat16 if _device.type=='cuda' and torch.cuda.is_bf16_supported() else (torch.float16 if _device.type=='cuda' else torch.float32)

        export_tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3.5-0.8B', use_fast=True)
        if export_tokenizer.pad_token_id is None:
            export_tokenizer.pad_token = export_tokenizer.eos_token

        export_model = AutoModelForCausalLM.from_pretrained(
            'Qwen/Qwen3.5-0.8B',
            dtype=_dtype if _device.type=='cuda' else torch.float32,
            low_cpu_mem_usage=True,
        ).to(_device)

        replace_ffns_with_fly_v2(export_model, cfg, adjacency)
        incompatible = export_model.load_state_dict(ckpt, strict=False)
        missing = [k for k in incompatible.missing_keys if '.mlp.' in k or k.startswith('flyffn_shared_graph.')]
        if missing:
            raise RuntimeError('Training checkpoint missing FlyFFN keys: '+str(missing[:20]))
        assert_qwen35_flyffn_v2(export_model, cfg.anchor_every)
        export_model.eval()
        print('Reconstructed FlyFFN-v2 from saved training checkpoint.')

    manifest = export_standalone(
        export_model,
        export_tokenizer,
        cfg,
        STANDALONE_DIR,
        metadata=report,
        artifacts_dir=OUTPUT_DIR,
    )
    print('✓ Repaired standalone package created')

    if UPLOAD_TO_HF:
        url = upload_standalone(
            STANDALONE_DIR,
            HF_REPO_ID,
            token=os.environ.get('HF_TOKEN'),
            private=HF_PRIVATE,
        )
        print('✓ Uploaded repaired standalone:', url)

manifest = json.loads(manifest_path.read_text())
assert manifest.get('verified') is True
assert state_path.exists()

print('\n✓ Standalone model verified')
print('Folder      :', STANDALONE_DIR)
print('State keys  :', manifest['state_keys'])
print('FlyFFN keys :', manifest['flyffn_state_keys'])
print('Size        :', f"{manifest['state_bytes']/1024**3:.3f} GB")
if UPLOAD_TO_HF:
    print('Hugging Face:', f'https://huggingface.co/{HF_REPO_ID}')



✓ Standalone model verified
Folder      : /content/TinyCeNN-LM/results/flyffn_v2_qwen35_08b/standalone
State keys  : 448
FlyFFN keys : 181
Size        : 1.884 GB
Hugging Face: https://huggingface.co/vtava/Qwen35-0.8B-FlyFFN-v2


In [9]:
#@title 9. Fresh standalone reload + chat test (no base-model download)
import gc, torch
from tinycenn_lm.qwen35_standalone import load_standalone

# Free comparison models if they exist to avoid GPU OOM.
for _name in ['qwen_model', 'fly_model']:
    if _name in globals():
        try:
            del globals()[_name]
        except Exception:
            pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

standalone_model, standalone_tokenizer = load_standalone(
    OUTPUT_DIR / 'standalone',
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

messages=[{'role':'user','content':'Explain in simple terms what a neural network is.'}]
prompt=standalone_tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
inputs=standalone_tokenizer(prompt,return_tensors='pt').to(next(standalone_model.parameters()).device)

with torch.inference_mode():
    out=standalone_model.generate(
        **inputs,
        max_new_tokens=160,
        do_sample=False,
        use_cache=True,
        pad_token_id=standalone_tokenizer.eos_token_id,
    )

answer=standalone_tokenizer.decode(
    out[0,inputs.input_ids.shape[1]:],
    skip_special_tokens=True,
).strip()

print('✓ Fresh standalone reconstruction succeeded')
print('\nFlyFFN-v2 standalone answer:\n', answer)


[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


✓ Fresh standalone reconstruction succeeded

FlyFFN-v2 standalone answer:
 A **neural network** is a computer system that learns how to make decisions based on data. It works by connecting different parts of the computer together to create a "brain."

Here is a simple explanation of how it works:

### 1. The "Brain" (The Layers)
Imagine you have a computer that can do math, but it doesn't understand what you are thinking. A neural network acts like a brain with three main parts:
*   **Input Layer:** This is where you put in the data (like a picture or a number).
*   **Hidden Layer:** This is where the brain processes the information.
*   **Output Layer:** This is where the brain gives the final answer (like a prediction or a decision).

### 2


In [10]:
#@title 10. Sync latest FastEval/results to the Hugging Face model repo
import json, shutil, os
from pathlib import Path
from tinycenn_lm.qwen35_standalone import upload_standalone

STANDALONE_DIR = OUTPUT_DIR / 'standalone'

# If FastEval was run, include it in the model repository.
if (OUTPUT_DIR/'fast_eval_50_qwen35.csv').exists():
    shutil.copy2(
        OUTPUT_DIR/'fast_eval_50_qwen35.csv',
        STANDALONE_DIR/'fast_eval_50_qwen35.csv'
    )
    print('✓ Added latest FastEval CSV')

if 'fast_eval_df' in globals():
    fast_eval_df.to_csv(STANDALONE_DIR/'fast_eval_50_qwen35.csv')

if UPLOAD_TO_HF:
    url = upload_standalone(
        STANDALONE_DIR,
        HF_REPO_ID,
        token=os.environ.get('HF_TOKEN'),
        private=HF_PRIVATE,
    )
    print('✓ Hugging Face synchronized:', url)
else:
    print('UPLOAD_TO_HF=False — standalone model remains local at', STANDALONE_DIR)


✓ Added latest FastEval CSV
✓ Hugging Face synchronized: https://huggingface.co/vtava/Qwen35-0.8B-FlyFFN-v2
